# 🛡️ Phase 6: Independent Evaluation Environment on Google Colab

Môi trường thực thi độc lập này đóng vai trò nạp mô hình đã fine-tune từ Hugging Face Hub (`thong7d/vihsd-xlmr-hate-speech`) và chạy đánh giá (Phase 6) trên tập dữ liệu kiểm tra của ViHSD.

### ✨ Đặc điểm thiết kế:
1. **Giải phóng Drive (Local-first)**: Không cần liên kết Google Drive, kết quả được tạo ngay tại bộ nhớ đệm và nén gửi về trình duyệt của bạn.
2. **Tương thích hoàn toàn (Tokenizer Sync)**: Tắt hoàn toàn cấu hình tách từ (`use_word_segmentation=False`) đồng bộ hoàn toàn với băm subword SentencePiece của XLM-RoBERTa nhằm bảo toàn thông tin ngữ cảnh tự nhiên.
3. **Tách biệt cấu hình (Modularity)**: Truyền trực tiếp repo ID thông qua CLI argument `--hf_repo_id` mà không làm thay đổi các file config huấn luyện.

### 🚀 Bước 1: Cấu hình môi trường và Tải mã nguồn

In [1]:
# Cấu hình thư mục cache cho Hugging Face
import os, sys
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf_datasets_cache"

# THÊM MỚI: Tự động nạp HF_TOKEN từ Colab Secrets để xác thực quyền truy cập repo Private
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        from huggingface_hub import login
        login(token=token, add_to_git_credential=True)
        print("🔑 Đăng nhập Hugging Face Hub thành công!")
except Exception as e:
    print(f"⚠️ Cảnh báo xác thực (Có thể repo đang để công khai): {e}")

REPO_URL  = "https://github.com/thong7d/hate-speech-detection.git"
REPO_NAME = "hate-speech-detection"
REPO_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_PATH):
    print(f"📥 Đang clone repository từ: {REPO_URL}...")
    os.system(f"git clone {REPO_URL} {REPO_PATH}")
else:
    print("🔄 Cập nhật mã nguồn mới nhất từ GitHub...")
    os.system(f"cd {REPO_PATH} && git pull origin main")

print("✅ Tải mã nguồn thành công!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


🔑 Đăng nhập Hugging Face Hub thành công!
📥 Đang clone repository từ: https://github.com/thong7d/hate-speech-detection.git...
✅ Tải mã nguồn thành công!


### 📦 Bước 2: Cài đặt thư viện dependencies

In [2]:
import os
import sys

print("📥 Đang cài đặt thư viện dependencies từ requirements.txt...")
os.system(f"pip install -q -r {REPO_PATH}/requirements.txt")

print("📥 Đang cài đặt thư viện tách từ tiếng Việt underthesea...")
os.system("pip install -q underthesea")

print("📥 Đang cài đặt dự án ở chế độ phát triển (editable mode)...")
os.system(f"pip install -q -e {REPO_PATH}")

# SỬA ĐỔI: Thêm REPO_PATH (thư mục gốc chứa gói src) vào đường dẫn hệ thống
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print("✅ Cài đặt môi trường thành công!")

📥 Đang cài đặt thư viện dependencies từ requirements.txt...
📥 Đang cài đặt thư viện tách từ tiếng Việt underthesea...
📥 Đang cài đặt dự án ở chế độ phát triển (editable mode)...
✅ Cài đặt môi trường thành công!


### 📊 Bước 3: Tải dữ liệu và Tiền xử lý tập Test (XLM-RoBERTa Tokenizer Sync)

In [3]:
from src.data.download import download_and_extract
from src.data.preprocessing import process_split

print("📥 Đang tải dữ liệu gốc từ GitHub...")
download_and_extract(f"{REPO_PATH}/configs/paths.yaml")

print("⚙️ Đang thực hiện tiền xử lý tập test (use_word_segmentation=False)...")
process_split(
    input_path=f"{REPO_PATH}/data/raw/vihsd/test.csv",
    output_path=f"{REPO_PATH}/data/processed/test.parquet",
    use_word_segmentation=False
)

print("✅ Tiền xử lý dữ liệu kiểm tra thành công! Tập test đã lưu dưới dạng Parquet.")

📥 Đang tải dữ liệu gốc từ GitHub...
Download successful.
Extracting files...
[OK] Extracted: test.csv -> /content/hate-speech-detection/data/raw/vihsd/test.csv
[OK] Extracted: dev.csv -> /content/hate-speech-detection/data/raw/vihsd/dev.csv
[OK] Extracted: train.csv -> /content/hate-speech-detection/data/raw/vihsd/train.csv
[SUCCESS] Phase 1: Data Acquisition completed successfully.
⚙️ Đang thực hiện tiền xử lý tập test (use_word_segmentation=False)...
✅ Tiền xử lý dữ liệu kiểm tra thành công! Tập test đã lưu dưới dạng Parquet.


### 🧠 Bước 4: Thực hiện đánh giá mô hình qua CLI

In [4]:
import os, shutil
# Tối ưu hóa phi trạng thái: Dọn dẹp kết quả và file zip cũ trước khi đánh giá mới
results_dir = f"{REPO_PATH}/results"
if os.path.exists(results_dir):
    shutil.rmtree(results_dir)
os.makedirs(results_dir, exist_ok=True)

zip_file = "/content/evaluation_results.zip"
if os.path.exists(zip_file):
    os.remove(zip_file)

print("🚀 Bắt đầu quá trình đánh giá mô hình thong7d/vihsd-xlmr-hate-speech...")
# Đánh giá mô hình nạp từ HF Hub và ghi đè tách từ là False
!cd /content/hate-speech-detection && python -m src.evaluation.evaluate --config configs/train.yaml --hf_repo_id thong7d/vihsd-xlmr-hate-speech --use_word_segmentation False

🚀 Bắt đầu quá trình đánh giá mô hình thong7d/vihsd-xlmr-hate-speech...
/content/hate-speech-detection/src/evaluation/evaluate.py:13: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  if hasattr(np, "core") and hasattr(np.core, "multiarray") and hasattr(np.core.multiarray, "_reconstruct"):
/content/hate-speech-detection/src/evaluation/evaluate.py:14: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can c

In [5]:
from huggingface_hub import list_repo_files
try:
    # Đã cập nhật repo_id sang mô hình XLM-R mới để đồng bộ hệ thống
    files = list_repo_files(repo_id="thong7d/vihsd-xlmr-hate-speech")
    print("📂 Các file hiện có trên HF Hub:")
    for f in files:
        print(f" - {f}")
except Exception as e:
    print(f"❌ Lỗi quét repo: {e}")

📂 Các file hiện có trên HF Hub:
 - .gitattributes
 - README.md
 - classifier.py
 - config.json
 - label_mapping.json
 - last-checkpoint/config.json
 - last-checkpoint/model.safetensors
 - last-checkpoint/optimizer.pt
 - last-checkpoint/rng_state.pth
 - last-checkpoint/scheduler.pt
 - last-checkpoint/sentencepiece.bpe.model
 - last-checkpoint/special_tokens_map.json
 - last-checkpoint/tokenizer.json
 - last-checkpoint/tokenizer_config.json
 - last-checkpoint/trainer_state.json
 - last-checkpoint/training_args.bin
 - metadata.json
 - metrics.json
 - model.safetensors
 - model/classifier.py
 - model/config.json
 - model/configuration_xlm_roberta.py
 - model/model.safetensors
 - model/sentencepiece.bpe.model
 - model/special_tokens_map.json
 - model/tokenizer.json
 - model/tokenizer_config.json
 - model/training_args.bin
 - model_card.md
 - sentencepiece.bpe.model
 - special_tokens_map.json
 - tokenizer.json
 - tokenizer_config.json
 - training_args.bin


### 📥 Bước 5: Nén và Tải kết quả về máy cục bộ

In [6]:
import shutil
import os
from google.colab import files

# Schema guard: Đảm bảo thư mục tồn tại trên ổ đĩa trước khi ra lệnh nén
os.makedirs(f"{REPO_PATH}/results", exist_ok=True)

print("📦 Đang nén thư mục kết quả đánh giá (results/)...")
shutil.make_archive("/content/evaluation_results", "zip", f"{REPO_PATH}/results")

print("📥 Đang tải file evaluation_results.zip về máy cục bộ của bạn...")
files.download("/content/evaluation_results.zip")
print("✅ Đã kích hoạt hộp thoại tải xuống thành công!")

📦 Đang nén thư mục kết quả đánh giá (results/)...
📥 Đang tải file evaluation_results.zip về máy cục bộ của bạn...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Đã kích hoạt hộp thoại tải xuống thành công!


### 📝 Bước 6: Hiển thị báo cáo đánh giá trực tiếp

In [7]:
from IPython.display import Markdown, display

report_path = f"{REPO_PATH}/results/evaluation_report.md"
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        content = f.read()
    print("\n" + "="*40 + " BÁO CÁO ĐÁNH GIÁ CHI TIẾT " + "="*40 + "\n")
    display(Markdown(content))
else:
    print("⚠️ Không tìm thấy tệp báo cáo evaluation_report.md!")


======================================== BÁO CÁO ĐÁNH GIÁ CHI TIẾT ========================================



# Evaluation Report

- Accuracy: 0.3425
- Macro F1: 0.2209
- Weighted F1: 0.4263
- Critical F1: 0.0824
- Offensive Priority F1: 0.0892
- Balanced Critical F1: 0.1101
- AUC-ROC Macro: None

## Per-Class Metrics

| Class | Precision | Recall | F1 | AUC-ROC |
|---|---:|---:|---:|---:|
| CLEAN | 0.7615 | 0.3700 | 0.4980 | N/A |
| OFFENSIVE | 0.0556 | 0.4662 | 0.0994 | N/A |
| HATE | 0.1304 | 0.0436 | 0.0654 | N/A |

## Confusion Matrix

```text
[[2028, 3263, 190], [227, 207, 10], [408, 250, 30]]
```

## Error Analysis Summary

- Total errors: 4348 / 6613
- Error rate: 0.6575
- False positive (CLEAN → toxic): 3453
- False negative (OFFENSIVE → CLEAN): 227
- False negative (HATE → CLEAN): 408
- OFFENSIVE ↔ HATE confusion: 260

## Error Counts

| Expected | Missed as CLEAN | Predicted as toxic from CLEAN |
|---|---:|---:|
| CLEAN | 0 | 0 |
| OFFENSIVE | 227 | 3263 |
| HATE | 408 | 190 |
